# Core Debug
> Lightweight project-wide logging helpers.

Usage:

    from cogitarelink.core.debug import get_logger, silence_logs, set_loglevel
    
    log = get_logger("graph")
    log.debug("hello")
    
    with silence_logs():
        ...  # all cogitarelink.* logs muted temporarily

In [ ]:
#| default_exp core.debug


In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
import logging, os, sys, contextlib
from typing import Iterator


In [ ]:
#| export
_FMT = "%(levelname).1s|%(name)s|%(message)s"
_ENV = "COG_LOGLEVEL"                   # env var to override
_ROOT = "cogitarelink"                  # prefix for all lib loggers


In [ ]:
#| export
# optional colour -----------------------------------------------------
try:
    from colorama import init as _c_init, Fore, Style
    _c_init()
    _COLOUR = {
        "DEBUG":   Fore.BLUE,
        "INFO":    Fore.GREEN,
        "WARNING": Fore.YELLOW,
        "ERROR":   Fore.RED,
        "CRITICAL":Fore.MAGENTA,
    }
    def _colour_fmt(record, fmt=_FMT):
        col = _COLOUR.get(record.levelname, "")
        reset = Style.RESET_ALL
        return f"{col}{logging.Formatter(fmt).format(record)}{reset}"
    _FORMATTER = logging.Formatter()    # dummy; replaced per-handler
    _COLOURIZED = True
except ImportError:                     # no colourama
    _FORMATTER = logging.Formatter(_FMT)
    _COLOURIZED = False


In [ ]:
#| export
def _new_handler():
    h = logging.StreamHandler(sys.stderr)
    if _COLOURIZED:
        h.setFormatter(logging.Formatter(_FMT))
        # colour via override of emit
        orig = h.emit
        def emit(rec):
            rec.msg = _colour_fmt(rec)
            rec.args = ()
            orig(rec)
        h.emit = emit           # type: ignore
    else:
        h.setFormatter(_FORMATTER)
    return h

In [ ]:
#| export
def get_logger(name: str, level: str | None = None) -> logging.Logger:
    full = f"{_ROOT}.{name}"
    log  = logging.getLogger(full)

    root_level = (level or os.getenv(_ENV) or "WARNING").upper()  # <-- keep
    if not log.handlers:
        log.addHandler(_new_handler())
        log.setLevel(root_level)          # ← first time we definitely set
        log.propagate = False
    else:
        # logger already exists; set level only if:
        #  • caller passed an explicit level  OR
        #  • level is still NOTSET (0)
        if level or log.level == logging.NOTSET:
            log.setLevel(root_level)

    return log

In [ ]:
#| export
def set_loglevel(level: str):
    "Globally set level for **all** cogitarelink.* loggers."
    root = logging.getLogger(_ROOT)
    root.setLevel(level.upper())
    for l in logging.Logger.manager.loggerDict.values():  # type: ignore
        if isinstance(l, logging.Logger) and l.name.startswith(_ROOT):
            l.setLevel(level.upper())

In [ ]:
#| export
@contextlib.contextmanager
def silence_logs(lvl="CRITICAL") -> Iterator[None]:
    olds={n:l.level for n,l in list(logging.Logger.manager.loggerDict.items()) if isinstance(l,logging.Logger) and n.startswith(_ROOT)}
    for n,l in list(logging.Logger.manager.loggerDict.items()):
        if isinstance(l,logging.Logger) and n.startswith(_ROOT): l.setLevel(lvl)
    try: yield
    finally:
        for n,l in list(logging.Logger.manager.loggerDict.items()):
            if isinstance(l,logging.Logger) and n in olds: l.setLevel(olds[n])

In [ ]:
#| hide
from fastcore.test import *
log1 = get_logger("demo")
assert log1.level != logging.NOTSET, "Logger level should not be NOTSET"
log2 = get_logger("demo")
test_is(log1, log2); test_eq(log1.level, logging.WARNING)
get_logger("demo", "INFO"); test_eq(log1.level, logging.INFO)
with silence_logs(): test_eq(log1.level, logging.CRITICAL)
test_eq(log1.level, logging.INFO)

In [ ]:
# from cogitarelink.core.debug import get_logger, silence_logs, set_loglevel
l=get_logger("demo","DEBUG")
l.debug("debug msg"); l.info("info msg"); l.warning("warning msg"); l.error("error msg"); l.critical("critical msg")
print("normal output")
with silence_logs():
  l.info("this should be silenced")
print("after silence_logs"); l.info("back to normal")

D|cogitarelink.demo|D|cogitarelink.demo|debug msg
I|cogitarelink.demo|I|cogitarelink.demo|info msg
W|cogitarelink.demo|W|cogitarelink.demo|warning msg
E|cogitarelink.demo|E|cogitarelink.demo|error msg
C|cogitarelink.demo|C|cogitarelink.demo|critical msg


normal output
after silence_logs


I|cogitarelink.demo|I|cogitarelink.demo|back to normal


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()